In [3]:
# ============================================================
#  AA1189 — 3D Flight Visualization with Plotly
#  Shows lat / lon / altitude on an interactive Mapbox globe
# ============================================================

import pandas as pd
import plotly.graph_objects as go
import numpy as np

# ── 1. LOAD & PREPARE DATA ───────────────────────────────────
CSV_PATH = r"C:\Users\Dev Sharma\OneDrive\Documents\MDP\Datasets\Single_AA1189.csv"
df = pd.read_csv(CSV_PATH)

# Parse lat/lon
df[['lat', 'lon']] = df['Position'].str.split(',', expand=True).astype(float)
df['UTC'] = pd.to_datetime(df['UTC'])
df = df.sort_values('UTC').reset_index(drop=True)

# ── Altitude column ──────────────────────────────────────────
# Try common column names for altitude — adjust if yours is different
alt_col = None
for candidate in ['Altitude', 'altitude', 'Alt', 'alt', 'FL', 'Level']:
    if candidate in df.columns:
        alt_col = candidate
        break

if alt_col:
    df['alt_ft'] = pd.to_numeric(df[alt_col], errors='coerce').fillna(0)
    print(f"Using altitude column: '{alt_col}'")
    print(f"Altitude range: {df['alt_ft'].min():.0f} ft → {df['alt_ft'].max():.0f} ft")
else:
    # Simulate a realistic cruise arc if no altitude column found
    print("No altitude column found — simulating altitude arc")
    n = len(df)
    # Bell curve: climb → cruise → descent
    x = np.linspace(0, np.pi, n)
    df['alt_ft'] = (np.sin(x) * 35000).clip(0)

print(f"Dataset: {len(df)} rows\n{df[['lat','lon','alt_ft']].head()}")

# ── 2. COLOR SCALE BY ALTITUDE ───────────────────────────────
# Normalize altitude 0-1 for coloring
alt_norm = (df['alt_ft'] - df['alt_ft'].min()) / (df['alt_ft'].max() - df['alt_ft'].min() + 1)

# ── 3. BUILD PLOTLY FIGURE ───────────────────────────────────
fig = go.Figure()

# --- 3a. Full 3D scatter trace (the flight path) ---
fig.add_trace(go.Scattergeo(
    lat=df['lat'],
    lon=df['lon'],
    mode='lines+markers',
    line=dict(width=2, color='rgba(100,180,255,0.4)'),
    marker=dict(
        size=4,
        color=df['alt_ft'],
        colorscale=[
            [0.0,  '#1a0a3d'],   # dark purple  — ground / low
            [0.3,  '#0d47a1'],   # deep blue    — climb
            [0.6,  '#00bcd4'],   # cyan         — cruise
            [0.85, '#00e5ff'],   # bright cyan  — high cruise
            [1.0,  '#ffffff'],   # white        — peak altitude
        ],
        colorbar=dict(
            title=dict(text='Altitude (ft)', font=dict(color='white', size=13)),
            tickfont=dict(color='white'),
            thickness=14,
            len=0.6,
            x=1.01,
        ),
        showscale=True,
        opacity=0.9,
    ),
    text=[
        f"Step {i}<br>lat: {r.lat:.4f}<br>lon: {r.lon:.4f}<br>alt: {r.alt_ft:,.0f} ft"
        for i, r in df.iterrows()
    ],
    hoverinfo='text',
    name='Flight Path',
))

# --- 3b. Start marker ---
fig.add_trace(go.Scattergeo(
    lat=[df['lat'].iloc[0]],
    lon=[df['lon'].iloc[0]],
    mode='markers+text',
    marker=dict(size=14, color='#00ff88', symbol='circle', line=dict(color='white', width=2)),
    text=['  ✈ Departure'],
    textfont=dict(color='#00ff88', size=13),
    textposition='middle right',
    hoverinfo='text',
    hovertext=f"DEPARTURE<br>{df['lat'].iloc[0]:.4f}, {df['lon'].iloc[0]:.4f}",
    name='Start',
))

# --- 3c. End marker ---
fig.add_trace(go.Scattergeo(
    lat=[df['lat'].iloc[-1]],
    lon=[df['lon'].iloc[-1]],
    mode='markers+text',
    marker=dict(size=14, color='#ff4444', symbol='square', line=dict(color='white', width=2)),
    text=['  🏁 Arrival  '],
    textfont=dict(color='#ff4444', size=13),
    textposition='middle right',
    hoverinfo='text',
    hovertext=f"ARRIVAL<br>{df['lat'].iloc[-1]:.4f}, {df['lon'].iloc[-1]:.4f}",
    name='End',
))

# ── 4. GEO LAYOUT — Dark Globe Centered on the Flight ────────
mid_lat = (df['lat'].max() + df['lat'].min()) / 2
mid_lon = (df['lon'].max() + df['lon'].min()) / 2

fig.update_geos(
    # 3D globe projection — rotation centers the view on the flight
    projection=dict(
        type='orthographic',
        rotation=dict(lon=mid_lon, lat=mid_lat),
    ),
    center=dict(lat=mid_lat, lon=mid_lon),

    # Land / water styling
    showland=True,
    landcolor='#0d1b2a',
    showocean=True,
    oceancolor='#050d1a',
    showlakes=True,
    lakecolor='#0a1628',
    showcountries=True,
    countrycolor='rgba(100,150,220,0.4)',
    countrywidth=0.8,
    showcoastlines=True,
    coastlinecolor='rgba(100,160,240,0.6)',
    coastlinewidth=1,
    showrivers=False,

    # Zoom to flight bounding box + padding
    lonaxis=dict(range=[df['lon'].min() - 3, df['lon'].max() + 3]),
    lataxis=dict(range=[df['lat'].min() - 3, df['lat'].max() + 3]),
)

# ── 5. FIGURE LAYOUT ─────────────────────────────────────────
fig.update_layout(
    title=dict(
        text='✈  AA1189 — 3D Flight Trajectory',
        font=dict(size=22, color='#b0d4ff', family='monospace'),
        x=0.5, xanchor='center', y=0.97,
    ),
    paper_bgcolor='#04080f',
    geo_bgcolor='#04080f',
    legend=dict(
        font=dict(color='white', size=12),
        bgcolor='rgba(10,20,40,0.8)',
        bordercolor='rgba(100,150,220,0.3)',
        borderwidth=1,
        x=0.01, y=0.99,
    ),
    margin=dict(l=0, r=0, t=50, b=0),
    height=720,
)

# ── 6. ALTITUDE PROFILE INSET (bonus mini chart) ─────────────
# Add a small altitude-over-time annotation using shapes
fig.add_annotation(
    text=(
        f"<b>Flight Stats</b><br>"
        f"Points: {len(df)}<br>"
        f"Max Alt: {df['alt_ft'].max():,.0f} ft<br>"
        f"Start: {df['lat'].iloc[0]:.2f}°N, {df['lon'].iloc[0]:.2f}°<br>"
        f"End:   {df['lat'].iloc[-1]:.2f}°N, {df['lon'].iloc[-1]:.2f}°"
    ),
    xref='paper', yref='paper',
    x=0.01, y=0.15,
    align='left',
    showarrow=False,
    font=dict(color='#80b8ff', size=11, family='monospace'),
    bgcolor='rgba(10,20,40,0.75)',
    bordercolor='rgba(80,140,220,0.4)',
    borderwidth=1,
    borderpad=8,
)

# ── 7. SAVE & SHOW ───────────────────────────────────────────
output_html = 'flight_3d.html'
fig.write_html(output_html)
print(f"Saved interactive 3D map → {output_html}")
print("Open flight_3d.html in your browser — drag to rotate the globe!")

import webbrowser, os
webbrowser.open('file://' + os.path.abspath('flight_3d.html'))


Using altitude column: 'Altitude'
Altitude range: 0 ft → 37025 ft
Dataset: 600 rows
         lat        lon  alt_ft
0  36.128887 -86.671219       0
1  36.128342 -86.671906       0
2  36.127609 -86.677513       0
3  36.127888 -86.677864       0
4  36.128208 -86.678284       0
Saved interactive 3D map → flight_3d.html
Open flight_3d.html in your browser — drag to rotate the globe!


True

In [4]:
df.columns

Index(['Timestamp', 'UTC', 'Callsign', 'Position', 'Altitude', 'Speed',
       'Direction', 'lat', 'lon', 'alt_ft'],
      dtype='str')